# Evaluación de una Red Neuronal

Entrenar una red es solo una parte del proceso.

Después necesitamos responder:

> **¿Qué tan bien funciona el modelo con observaciones nuevas?**

En este notebook aprenderemos a evaluar una red neuronal de clasificación.

## Objetivos

Al finalizar podrás:

- diferenciar training y test performance;
- calcular accuracy;
- construir una confusion matrix;
- interpretar precision, recall y F1-score;
- analizar errores individuales;
- entender por qué una sola métrica puede ser insuficiente.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn


## 1. Datos y modelo


In [ ]:
torch.manual_seed(42)

iris = load_iris()

X_train, X_test, y_train, y_test = train_test_split(
    iris.data,
    iris.target,
    test_size=0.20,
    random_state=42,
    stratify=iris.target
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 3)
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for epoch in range(250):
    logits = model(X_train_t)
    loss = criterion(logits, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


## 2. Modo de evaluación

Durante evaluación no necesitamos calcular gradientes.

Usamos:

```python
with torch.no_grad():
```

Esto reduce trabajo innecesario.


In [ ]:
with torch.no_grad():
    train_logits = model(X_train_t)
    test_logits = model(X_test_t)

    train_pred = torch.argmax(train_logits, dim=1)
    test_pred = torch.argmax(test_logits, dim=1)


## 3. Accuracy

\[
\mathrm{Accuracy}
=
\frac{\text{predicciones correctas}}
{\text{total de predicciones}}.
\]


In [ ]:
train_accuracy = (train_pred == y_train_t).float().mean()
test_accuracy = (test_pred == y_test_t).float().mean()

print(f"Training accuracy: {train_accuracy.item():.3f}")
print(f"Test accuracy:     {test_accuracy.item():.3f}")


### Pregunta

¿Por qué nos interesa especialmente el test accuracy?

<details>
<summary><strong>Mostrar solución</strong></summary>

Porque el test set contiene observaciones que no fueron utilizadas para ajustar los parámetros.

Por eso nos da una mejor idea de la capacidad de **generalización** del modelo.

</details>


## 4. Confusion Matrix

Accuracy nos da un solo número.

Pero queremos saber:

> **¿Qué clases está confundiendo el modelo?**


In [ ]:
cm = confusion_matrix(
    y_test_t.numpy(),
    test_pred.numpy()
)

print(cm)


La confusion matrix se interpreta como:

```text
                   Predicted
                class 0  class 1  class 2

True class 0
True class 1
True class 2
```

La diagonal representa predicciones correctas.


In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.xticks(range(3), iris.target_names)
plt.yticks(range(3), iris.target_names)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("Confusion Matrix")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.show()


### Pregunta

¿Qué dos especies suelen ser más difíciles de separar en Iris?

<details>
<summary><strong>Pista</strong></summary>

Recuerda los gráficos de petal length vs. petal width.

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

Generalmente **Versicolor** y **Virginica** presentan más solapamiento que Setosa.

La confusion matrix permite comprobar si el modelo también refleja esa dificultad.

</details>


## 5. Precision, Recall y F1

Para cada clase podemos calcular otras métricas.

### Precision

\[
\mathrm{Precision}
=
\frac{TP}{TP+FP}
\]

Pregunta:

> De los ejemplos que el modelo clasificó como esta clase, ¿cuántos eran realmente de esa clase?

### Recall

\[
\mathrm{Recall}
=
\frac{TP}{TP+FN}
\]

Pregunta:

> De todos los ejemplos reales de esta clase, ¿cuántos encontró el modelo?

### F1-score

Combina precision y recall.


In [ ]:
print(
    classification_report(
        y_test_t.numpy(),
        test_pred.numpy(),
        target_names=iris.target_names
    )
)


## 6. Analizando errores

Veamos qué observaciones fueron clasificadas incorrectamente.


In [ ]:
incorrect = test_pred != y_test_t

incorrect_indices = torch.where(incorrect)[0]

print("Número de errores:", len(incorrect_indices))
print("Índices:", incorrect_indices.tolist())


In [ ]:
for idx in incorrect_indices.tolist():

    true_class = iris.target_names[y_test_t[idx].item()]
    pred_class = iris.target_names[test_pred[idx].item()]

    print(
        f"Sample {idx:2d} | "
        f"true={true_class:10s} | "
        f"pred={pred_class:10s}"
    )


Esta parte es muy importante científicamente.

No basta con preguntar:

> ¿Cuál es la accuracy?

También debemos investigar:

> ¿Dónde falla el modelo y por qué?


## 7. Probabilidades del modelo

Podemos convertir logits a probabilidades.


In [ ]:
softmax = nn.Softmax(dim=1)

with torch.no_grad():
    probabilities = softmax(test_logits)

for i in range(min(5, len(probabilities))):
    print(
        f"Sample {i}:",
        probabilities[i].numpy(),
        "prediction:",
        iris.target_names[test_pred[i].item()]
    )


### Pregunta

¿Qué diferencia existe entre:

```text
[0.98, 0.01, 0.01]
```

y:

```text
[0.10, 0.46, 0.44]
```

aunque ambas produzcan una clase predicha?

<details>
<summary><strong>Mostrar solución</strong></summary>

La primera predicción muestra una separación muy fuerte entre clases.

La segunda está mucho más cerca de una decisión ambigua entre dos clases.

Por eso observar las probabilidades puede aportar información adicional sobre el comportamiento del modelo.

</details>


## 8. Accuracy no siempre es suficiente

Imagine un dataset científico donde:

```text
99% de los eventos → clase A
1% de los eventos  → clase B
```

Un modelo que siempre diga:

```text
A
```

tendría:

```text
99% accuracy
```

pero sería inútil para encontrar la clase B.

Por eso la evaluación debe adaptarse a la pregunta científica.


## 9. Ejercicio

Completa una función:


In [ ]:
def accuracy(model, X, y):
    # completa aquí
    pass


<details>
<summary><strong>Pista</strong></summary>

Necesitas:

```python
with torch.no_grad():
    logits = model(X)
    predictions = ...
```

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

```python
def accuracy(model, X, y):
    with torch.no_grad():
        logits = model(X)
        predictions = torch.argmax(logits, dim=1)
        return (predictions == y).float().mean().item()
```

</details>


# Para recordar

Una evaluación completa puede incluir:

```text
Test Loss
Accuracy
Confusion Matrix
Precision
Recall
F1-score
Individual Errors
Prediction Probabilities
```

La pregunta no es únicamente:

> **¿Qué tan alto es el score?**

También debemos preguntar:

> **¿En qué condiciones funciona y en cuáles falla?**


## Recursos

- [Scikit-learn — Classification Metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)
- [Scikit-learn — Confusion Matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
